# AI-Assisted Datamine Studio RM Workflow

_Generated by dmstudio.notebook_builder on 2026-07-18 21:16_

> Run with: `jupyter nbconvert --to notebook --execute --inplace ai_agent_workflow_tutorial.ipynb`

In [ ]:
# Auto-generated preamble
import warnings
warnings.filterwarnings("ignore")
print("Notebook: AI-Assisted Datamine Studio RM Workflow")

## 1. Setting Up the AI Assistant (MCP Server)

To allow an AI agent to see your project files, inspect command schemas, and generate Jupyter notebooks, add the `mcp_server.py` to your AI client configuration.

### For Claude Desktop or Antigravity Config:
```json
{
  "mcpServers": {
    "dmstudio": {
      "command": "C:\\path\\to\\dmstudio-rm\\.venv\\Scripts\\python.exe",
      "args": ["C:\\path\\to\\dmstudio-rm\\mcp_server.py"]
    }
  }
}
```


## 2. Interactive AI Chat Prompt Example

Copy and paste the following prompt into your AI agent's chat window to let it tailor the workflow dynamically:

> **Prompt:**
> "I want to create a workflow using the Datamine tutorial files. Look in `tutorials/Database/DMTutorials/Data/VBOP/Datamine/`.
> 1. Load `_vb_collars.dmx`, `_vb_surveys.dmx`, and `_vb_assays.dmx`.
> 2. Sort the assays using MGSORT by BHID and FROM.
> 3. Desurvey them using HOLES3D to get a 3D drillhole dataset.
> 4. Composite the desurveyed assays to 2.0 meter intervals using COMPDH.
> 5. Read the final composited table into Pandas, filter out null/negative grades, and plot the distribution of gold grades (AU) downhole using Seaborn."

### How the AI Agent Uses MCP Tools Under the Hood:
1. **Discovers Files**: The agent lists or searches your directory to find the absolute paths.
2. **Inspects Datamine Headers**: The agent runs `read_datamine_file` on `_vb_collars.dmx` to see the exact columns (detecting `BHID`, `XCOLLAR`, `YCOLLAR`, `ZCOLLAR`).
3. **Inspects Commands**: The agent calls `get_command_schema("holes3d")` and `get_command_schema("compdh")` to check the parameters.
4. **Builds Workflow**: The agent generates the tailored notebook dynamically based on your request.


## 3. Reference Implementation: Datamine & Pandas Round-Trip

Below is the complete reference implementation demonstrating the exact round-trip pipeline.
Run the cells below to execute this workflow.


### Step 3.1: Setup and Environment Verification
First, import the required packages and check that we can attach to Datamine Studio RM.


In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dmstudio import dmcommands, dm_io, sandbox

# Initialize commands
cmd = dmcommands.init()
print("Attached to Datamine Studio RM successfully!")


### Step 3.2: Prepare the Inputs
We'll copy the tutorial database files to the active project directory so Datamine can access them without absolute paths (preventing path parser issues).


In [ ]:
# Setup active project directory files
db_dir = r"D:\Active\dmstudio-rm\tutorials\Database\DMTutorials\Data\VBOP\Datamine"

# List of files we need
files_to_copy = ["_vb_collars.dmx", "_vb_surveys.dmx", "_vb_assays.dmx"]

for f in files_to_copy:
    src = os.path.join(db_dir, f)
    dst = os.path.basename(f) # Copies directly to active project directory
    if os.path.exists(src):
        # Read and save using dm_io to ensure correct format in workspace
        df = dm_io.read_datamine(src)
        dm_io.to_datamine(df, dst)
        print(f"Copied {f} to active project workspace as {dst}")


### Step 3.3: Sort Assays
We need to make sure the assay table is sorted correctly by BHID and FROM before desurveying or compositing.


In [ ]:
# Sort assays
cmd.mgsort(
    in_i="_vb_assays",
    out_o="assays_sorted",
    keys_f=["BHID", "FROM"]
)
print("Sorted assays written to assays_sorted")


### Step 3.4: Drillhole Desurveying (HOLES3D)
Now we run desurveying to generate 3D coordinates along the drillholes.


In [ ]:
# Run HOLES3D process
cmd.holes3d(
    in_i="_vb_collars",
    survey_i="_vb_surveys",
    assays_i="assays_sorted",
    out_o="desurveyed_holes",
    bhid_f="BHID",
    xcollar_f="XCOLLAR",
    ycollar_f="YCOLLAR",
    zcollar_f="ZCOLLAR",
    dipcollar_f="DIP",
    azicollar_f="AZIMUTH",
    declin_f="DIP",
    azimuth_f="AZIMUTH",
    at_f="AT",
    from_f="FROM",
    to_f="TO"
)
print("Desurveying complete! Output saved to desurveyed_holes.")


### Step 3.5: Downhole Compositing (COMPDH)
We composite the drillhole assays into regular 2.0 meter intervals.


In [ ]:
# Run COMPDH to composite grade intervals
cmd.compdh(
    in_i="desurveyed_holes",
    out_o="composited_holes",
    interval_p=2.0,
    bhid_f="BHID",
    grade_f="AU",
    from_f="FROM",
    to_f="TO"
)
print("Compositing complete! Output saved to composited_holes.")


### Step 3.6: Read back to Pandas & Visualize
Finally, read the composited table back into a Pandas DataFrame and plot the gold grade distribution.


In [ ]:
# Read composited table back to python
df_comp = dm_io.read_datamine("composited_holes.dm")

# Filter out negative values / missing values
df_clean = df_comp[df_comp["AU"] >= 0.0]

print(f"Loaded {len(df_clean)} composite intervals.")
print(df_clean[["BHID", "FROM", "TO", "AU"]].head(10))

# Plot grade vs. downhole depth (FROM)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x="FROM", y="AU", alpha=0.5, color="darkcyan")
sns.regplot(data=df_clean, x="FROM", y="AU", scatter=False, color="red")
plt.title("Gold Grade (AU) vs. Downhole Depth (FROM) - 2m Composites")
plt.xlabel("Downhole Depth (m)")
plt.ylabel("Gold Grade (g/t)")
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()
